# Day 2: Chunking Implementation

Goal: take the 5,183 SciFact abstracts saved in `corpus.json` and produce four parallel chunked versions of the corpus, one per strategy:

1. **Document-level** — one chunk per abstract (no splitting)
2. **Fixed-size** — split every N characters/tokens, no respect for boundaries
3. **Recursive character splitting** — LangChain's `RecursiveCharacterTextSplitter` at ~512 tokens
4. **Semantic chunking** — group sentences by embedding similarity

Each output chunk carries `{chunk_id, doc_id, text}` so we can score retrieval against ground truth on Day 4.

Outputs (one JSON file per strategy):
- `chunks_document.json`
- `chunks_fixed.json`
- `chunks_recursive.json`
- `chunks_semantic.json`

## 0. Install dependencies

Run once. The semantic chunker needs an embedding model — we use `bge-small-en-v1.5` here so it matches what we'll use on Day 3 for indexing.

In [ ]:
!pip install langchain langchain-community langchain-experimental langchain-huggingface sentence-transformers tiktoken

## 1. Load the cleaned corpus from Day 1

In [ ]:
import json
from pathlib import Path

with open("corpus.json") as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} documents")
print(f"Sample doc keys: {list(corpus[0].keys())}")
print(f"Sample title: {corpus[0]['title']}")

## 2. Helper: save chunks and report stats

All four strategies will produce a list of `{chunk_id, doc_id, text}` dicts. This helper writes them out and prints sanity-check stats.

In [ ]:
import numpy as np

def save_and_report(chunks, name, path):
    with open(path, "w") as f:
        json.dump(chunks, f)

    lengths = np.array([len(c["text"].split()) for c in chunks])
    docs_covered = len({c["doc_id"] for c in chunks})

    print(f"=== {name} ===")
    print(f"  Total chunks:    {len(chunks)}")
    print(f"  Unique docs:     {docs_covered} / {len(corpus)}")
    print(f"  Avg words/chunk: {lengths.mean():.1f}")
    print(f"  Median:          {np.median(lengths):.0f}")
    print(f"  Min / Max:       {lengths.min()} / {lengths.max()}")
    print(f"  Saved to:        {path}")
    print()

## 3. Strategy 1 — Document-level (no splitting)

Each abstract becomes a single chunk. Day 1 stats showed 99.4% of abstracts are under 512 words, so most will fit comfortably in a 512-token embedding context. This is the natural baseline.

In [ ]:
doc_chunks = []
for doc in corpus:
    # Prepend title — it carries strong topical signal and almost always fits
    text = f"{doc['title']}. {doc['abstract_text']}"
    doc_chunks.append({
        "chunk_id": f"{doc['doc_id']}_0",
        "doc_id": doc["doc_id"],
        "text": text,
    })

save_and_report(doc_chunks, "Document-level", "chunks_document.json")

## 4. Strategy 2 — Fixed-size chunking

Split every N characters, no regard for sentence/paragraph boundaries. This is the deliberately-naive baseline. We use `CharacterTextSplitter` with an unlikely separator so it falls back to pure length-based slicing.

Rough conversion: ~4 chars/token, so 512 tokens ≈ 2048 chars. We use a small overlap to avoid cutting mid-word at every boundary.

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    separator="",          # force pure length-based splits
    chunk_size=2048,       # ~512 tokens
    chunk_overlap=200,
    length_function=len,
)

fixed_chunks = []
for doc in corpus:
    text = f"{doc['title']}. {doc['abstract_text']}"
    pieces = fixed_splitter.split_text(text)
    for i, piece in enumerate(pieces):
        fixed_chunks.append({
            "chunk_id": f"{doc['doc_id']}_{i}",
            "doc_id": doc["doc_id"],
            "text": piece,
        })

save_and_report(fixed_chunks, "Fixed-size", "chunks_fixed.json")

## 5. Strategy 3 — Recursive character splitting (≈512 tokens)

LangChain's `RecursiveCharacterTextSplitter` tries paragraph → sentence → word → char in order, so chunks tend to land on natural boundaries. This is the standard go-to for RAG and the strategy your plan calls out specifically.

We size by tokens (not characters) using `tiktoken` for accuracy, since 512 tokens is the spec.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

recursive_chunks = []
for doc in corpus:
    text = f"{doc['title']}. {doc['abstract_text']}"
    pieces = recursive_splitter.split_text(text)
    for i, piece in enumerate(pieces):
        recursive_chunks.append({
            "chunk_id": f"{doc['doc_id']}_{i}",
            "doc_id": doc["doc_id"],
            "text": piece,
        })

save_and_report(recursive_chunks, "Recursive (512 tokens)", "chunks_recursive.json")

## 6. Strategy 4 — Semantic chunking

Splits at points where consecutive sentence embeddings diverge most — i.e. groups semantically-related sentences together. This is the slowest strategy because it has to embed every sentence during chunking.

We use `bge-small-en-v1.5` so the chunking embedding matches the indexing embedding we'll use on Day 3. Expect this cell to run for several minutes on CPU.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95,
)

semantic_chunks = []
for doc in tqdm(corpus, desc="Semantic chunking"):
    text = f"{doc['title']}. {doc['abstract_text']}"
    # Short abstracts (1–2 sentences) can break the splitter — pass them through whole
    if len(doc["abstract_sentences"]) < 3:
        pieces = [text]
    else:
        try:
            pieces = semantic_splitter.split_text(text)
        except Exception:
            pieces = [text]
    for i, piece in enumerate(pieces):
        semantic_chunks.append({
            "chunk_id": f"{doc['doc_id']}_{i}",
            "doc_id": doc["doc_id"],
            "text": piece,
        })

save_and_report(semantic_chunks, "Semantic", "chunks_semantic.json")

## 7. Side-by-side comparison

Quick at-a-glance summary so you can see how chunk counts and granularity differ across strategies. The strategy with the right balance — enough granularity to be precise, not so much that signal gets fragmented — is what we're hunting for on Day 4.

In [ ]:
summary = []
for name, path in [
    ("Document-level", "chunks_document.json"),
    ("Fixed-size",     "chunks_fixed.json"),
    ("Recursive 512",  "chunks_recursive.json"),
    ("Semantic",       "chunks_semantic.json"),
]:
    with open(path) as f:
        chunks = json.load(f)
    lengths = np.array([len(c["text"].split()) for c in chunks])
    summary.append({
        "Strategy":      name,
        "Total chunks":  len(chunks),
        "Chunks/doc":    round(len(chunks) / len(corpus), 2),
        "Avg words":     round(float(lengths.mean()), 1),
        "Median words":  int(np.median(lengths)),
        "Max words":     int(lengths.max()),
    })

import pandas as pd
pd.DataFrame(summary)

## Next: Day 3 — Indexing

With four chunked corpora saved, Day 3 is to embed each one with `bge-small-en-v1.5` and build four separate vector databases (FAISS or Chroma both work locally). Then Day 4 runs the 338 validation claims through each index and computes Hit Rate and MRR.